In [1]:
### set up the notebook
import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr
import numpy as np
import dask as da

from IPython.core.display import display, HTML
display(HTML("<style>.container { width:90% !important; }</style>"))
np.set_printoptions(linewidth=100) 

plt.rcParams.update({'font.size': 14})

da.config.set(**{'array.slicing.split_large_chunks': True})

In [9]:
### read in ACCESS and buoy loctaions

# ACCESS
access = xr.open_mfdataset('../data_to_publish/ACCESS_ZWD_bst_20230330_20230710.nc')

# buoy locations
buoy_loc = pd.read_csv('../data_to_publish/buoy_locations_FSP_latlon.csv')

# drop duplicate times
access = access.drop_duplicates('time')

In [11]:
### Interpolate model data to buoy locations

# ACCESS
sn40_ZWD_a = access.interp(lat=buoy_loc['lat'].values[0], lon=buoy_loc['lon'].values[0], method='linear')
sn20_ZWD_a = access.interp(lat=buoy_loc['lat'].values[1], lon=buoy_loc['lon'].values[1], method='linear')
sn06_ZWD_a = access.interp(lat=buoy_loc['lat'].values[2], lon=buoy_loc['lon'].values[2], method='linear')
ss05_ZWD_a = access.interp(lat=buoy_loc['lat'].values[3], lon=buoy_loc['lon'].values[3], method='linear')
ss20_ZWD_a = access.interp(lat=buoy_loc['lat'].values[4], lon=buoy_loc['lon'].values[4], method='linear')
ss30_ZWD_a = access.interp(lat=buoy_loc['lat'].values[5], lon=buoy_loc['lon'].values[5], method='linear')
ss40_ZWD_a = access.interp(lat=buoy_loc['lat'].values[6], lon=buoy_loc['lon'].values[6], method='linear')
swxt_ZWD_a = access.interp(lat=buoy_loc['lat'].values[7], lon=buoy_loc['lon'].values[7], method='linear')
sext_ZWD_a = access.interp(lat=buoy_loc['lat'].values[8], lon=buoy_loc['lon'].values[8], method='linear')

In [12]:
### Create a new dataset with the interpolated values

# ACCESS
access_at_buoys = xr.Dataset(data_vars={'sn40': ('time', sn40_ZWD_a.ZWD.values),
                                        'sn20': ('time', sn20_ZWD_a.ZWD.values),
                                        'sn06': ('time', sn06_ZWD_a.ZWD.values),
                                        'ss05': ('time', ss05_ZWD_a.ZWD.values),
                                        'ss20': ('time', ss20_ZWD_a.ZWD.values),
                                        'ss30': ('time', ss30_ZWD_a.ZWD.values),
                                        'ss40': ('time', ss40_ZWD_a.ZWD.values),
                                        'swxt': ('time', swxt_ZWD_a.ZWD.values),
                                        'sext': ('time', sext_ZWD_a.ZWD.values)},
                             coords={'time': access.time}, 
                             attrs={'info': 'ACCESS Zenith Wet Delay data interpolated to buoy locations.', 
                                   'acknowledgement': 'This research used the ACCESS-NRI’s model ACCESS-C infrastructure, which is enabled by the Australian Government’s National Collaborative Research Infrastructure Strategy (NCRIS).'})
    

In [13]:
### Save the datasets

# ACCESS
access_at_buoys.to_netcdf('../data_to_publish/ACCESS_at_buoys_FSP.nc')
